# Thermal MZI Simulation Analysis

This notebook documents the current understanding of the `thermal_mzi.nyancir` circuit
simulation using circulax, including the ThermalPhaseShifter model and the issues encountered.

## 1. Circuit Topology

```
                    ┌──────────────┐
                    │  X1 (heater) │
         W1  o1─────┤  forward     ├─────o2  W3
        ┌───────────┘  I1 drives   └───────────┐
        │              current                  │
  LS1 ──┤ X2 (mmi1x2)                    X4 (mmi2x2) ├── PD1 (W5)
  (W7)  │                                      │       ├── PD2 (W6)
        │                                      │
        └───────────┐              ┌───────────┘
         W2  o2─────┤  X3 (heater) ├─────o1  W4
                    │  FLIPPED 180°│
                    │  both pins   │
                    │  grounded    │
                    └──────────────┘
```

**Key observations:**
- X1 is not flipped: `transform = [1,0,0,1]` → light enters o1, exits o2 (forward)
- X3 IS flipped: `transform = [-1,0,0,-1]` → light enters o2, exits o1 (backward)
- Both X1 and X3 use the same `ThermalPhaseShifter` model
- X1 has current from I1 (0→60mA sweep), X3 has both heater terminals grounded (no current)

In [ ]:
import json
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from circulax.s_transforms import s_to_y

# Load and display circuit connectivity
with open('thermal_mzi.nyancir') as f:
    nyancir = json.load(f)

for key, data in nyancir.items():
    if isinstance(data, dict) and 'nets' in data:
        name = data.get('name', key.split(':')[-1])
        typ = data.get('type', '?')
        transform = data.get('transform', [1,0,0,1,0,0])
        nets = data.get('nets', {})
        flipped = transform[0] < 0 and transform[3] < 0
        print(f'{name:6s} ({typ:20s}) flipped={flipped}  nets={nets}')

## 2. ThermalPhaseShifter Model

The model computes a complex transmission coefficient T:

$$T = T_{\text{mag}} \cdot e^{-j\phi}$$

where:
- $T_{\text{mag}} = 10^{-\text{loss}/20}$ — amplitude from propagation loss
- $\phi = 2\pi \cdot n_{\text{eff}} \cdot L / \lambda + \Delta\phi$ — total phase
- $\Delta\phi = \pi \cdot \eta_{\pi/W} \cdot P_{\text{diss}}$ — thermal phase shift
- $P_{\text{diss}} = V_{\text{bias}}^2 / R$ — dissipated power

In [ ]:
# Model parameters
R_ohm = 120.0
eta_pi_per_W = 12.5
length_um = 100.0  # NOTE: nyancir has length=320 but model param is length_um
loss_dBcm = 1.0
neff0 = 2.4
wl = 1.55  # µm

# Derived quantities
loss_val = loss_dBcm * (length_um / 10000.0)
T_mag = 10.0 ** (-loss_val / 20.0)
phi_base = 2.0 * np.pi * (neff0 * (length_um / wl))

print(f'T_mag = {T_mag:.6f} (insertion loss = {loss_val:.4f} dB)')
print(f'phi_base = {phi_base:.2f} rad = {phi_base/(2*np.pi):.2f} cycles')
print(f'|T|² = {T_mag**2:.6f}')

# Current sweep
I_sweep = np.linspace(0, 0.06, 1001)
P_diss = I_sweep**2 * R_ohm
dphi = np.pi * eta_pi_per_W * P_diss

print(f'\nAt I=60mA: P_diss = {P_diss[-1]:.4f} W, dphi = {dphi[-1]:.2f} rad = {dphi[-1]/np.pi:.2f}π')
print(f'Number of fringes: {dphi[-1]/(2*np.pi):.1f}')

## 3. The VCVS Formulation (Original)

The original model uses a Voltage-Controlled Voltage Source constraint:

$$o_2 = T \cdot o_1$$

This works correctly for **forward** propagation (signal enters o1, exits o2):
- Output field = T × input field ✓
- |output| = T_mag × |input| ✓
- Phase delay = φ ✓

But for **backward** propagation (X3, signal enters o2, exits o1):
- The constraint gives: o1 = o2 / T
- |output| = |input| / T_mag → **GAIN instead of loss** ✗
- Phase advance = +φ instead of delay = −φ ✗

In [ ]:
# Demonstrate VCVS forward vs backward
T = T_mag * np.exp(-1j * (phi_base + dphi[500]))  # At I=30mA

# Forward: o2 = T * o1
o1_in = 1.0 + 0j  # unit input at o1
o2_out_fwd = T * o1_in
print(f'Forward (o1→o2):')
print(f'  |o2| = {abs(o2_out_fwd):.6f} (should be {T_mag:.6f})')
print(f'  phase(o2) = {np.angle(o2_out_fwd):.4f} rad')

# Backward: o1 = o2 / T
o2_in = 1.0 + 0j  # unit input at o2
o1_out_bwd = o2_in / T
print(f'\nBackward (o2→o1) with VCVS:')
print(f'  |o1| = {abs(o1_out_bwd):.6f} (should be {T_mag:.6f}, got {1/T_mag:.6f})')
print(f'  phase(o1) = {np.angle(o1_out_bwd):.4f} rad (INVERTED!)')

# What it SHOULD be for reciprocal:
o1_out_correct = T * o2_in
print(f'\nBackward CORRECT (reciprocal):')
print(f'  |o1| = {abs(o1_out_correct):.6f}')
print(f'  phase(o1) = {np.angle(o1_out_correct):.4f} rad')

## 4. The Y-Matrix Approach

To fix the backward direction, we tried converting S→Y:

$$S = \begin{pmatrix} 0 & T \\ T & 0 \end{pmatrix} \quad\rightarrow\quad Y = \frac{1}{1-T^2} \begin{pmatrix} 1+T^2 & -2T \\ -2T & 1+T^2 \end{pmatrix}$$

**Problem:** The Y-matrix entries depend on $1/(1-T^2)$, where $T^2 = |T|^2 e^{-2j\phi}$.
As $\phi$ rotates (with current), $1-T^2$ oscillates between ~0.002 and ~2, causing
the Y-matrix entries to swing between ~0.5 and ~500.

This causes non-physical **amplitude oscillations** at the nodes — the phase shifter
should only rotate the phase, not change the amplitude.

In [ ]:
# Y-matrix entries as a function of heater current
phi_total = phi_base + dphi
T_arr = T_mag * np.exp(-1j * phi_total)
T2_arr = T_arr**2
denom = 1 - T2_arr

Y11_arr = (1 + T2_arr) / denom
Y12_arr = -2 * T_arr / denom

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(I_sweep * 1e3, np.abs(Y11_arr), label='|Y11|', linewidth=0.8)
axes[0].plot(I_sweep * 1e3, np.abs(Y12_arr), label='|Y12|', linewidth=0.8)
axes[0].set_ylabel('|Y| magnitude')
axes[0].legend()
axes[0].set_title('Y-matrix entries oscillate wildly with heater current')
axes[0].set_yscale('log')

axes[1].plot(I_sweep * 1e3, np.abs(denom), linewidth=0.8, color='red')
axes[1].set_ylabel('|1 - T²|')
axes[1].set_xlabel('Heater current [mA]')
axes[1].set_title('Denominator 1-T² (goes near zero → Y blows up)')

plt.tight_layout()
plt.show()
print(f'Y11 range: {float(np.min(np.abs(Y11_arr))):.1f} to {float(np.max(np.abs(Y11_arr))):.1f}')
print(f'|1-T²| range: {float(np.min(np.abs(denom))):.4f} to {float(np.max(np.abs(denom))):.4f}')

## 5. MMI Models (Verified Correct)

### mmi1x2 (X2 — input splitter)
- Equal real-valued split: `S(o1,o2) = S(o1,o3) = amp / √2`
- No phase difference between arms → balanced MZI

### mmi2x2 (X4 — output combiner)
- Bar ports (o1→o3, o2→o4): real-valued `thru`
- Cross ports (o1→o4, o2→o3): imaginary `1j × cross`
- 90° phase between bar and cross → **complementary outputs**

For inputs $E_1, E_2$:
- PD1 (o3): $\propto |E_1 + jE_2|^2 \propto 1 + \sin(\Delta\phi)$
- PD2 (o4): $\propto |jE_1 + E_2|^2 \propto 1 - \sin(\Delta\phi)$

These are complementary: PD1 + PD2 = constant.

In [ ]:
import sax
import sax.models as sm

# Verify mmi2x2 S-matrix at 1.55 µm
s_dict = sm.mmi2x2(wl=1.55, wl0=1.55, fwhm=0.2, loss_dB=0.3)
print('mmi2x2 S-parameters at 1.55 µm:')
for (p1, p2), val in sorted(s_dict.items()):
    v = complex(val)
    if abs(v) > 1e-10:
        print(f'  S({p1},{p2}) = {v:.4f}  (|S|={abs(v):.4f}, phase={np.degrees(np.angle(v)):.1f}°)')

# Expected MZI output as function of phase difference
dphi_sweep = np.linspace(0, 4*np.pi, 1000)
thru = float(np.abs(complex(s_dict[('o1', 'o3')])))
cross = float(np.abs(complex(s_dict[('o1', 'o4')])))

# Input fields from splitter (equal amplitude, no phase diff at splitter)
E1 = 1.0 * np.exp(-1j * dphi_sweep)  # top arm with phase shift
E2 = 1.0 * np.ones_like(dphi_sweep)   # bottom arm (reference)

# After mmi2x2
PD1 = np.abs(thru * E1 + 1j * cross * E2)**2
PD2 = np.abs(1j * cross * E1 + thru * E2)**2

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(dphi_sweep / np.pi, PD1, label='PD1 (o3)', linewidth=1.5)
ax.plot(dphi_sweep / np.pi, PD2, label='PD2 (o4)', linewidth=1.5)
ax.set_xlabel('Phase difference Δφ [π rad]')
ax.set_ylabel('Detected power [a.u.]')
ax.set_title('Expected MZI output — complementary sinusoidal')
ax.legend()
plt.tight_layout()
plt.show()

## 6. The Fix: VCVS + Port Swap for Flipped Instances

The correct approach:

1. **Keep the VCVS formulation** — it correctly models constant-amplitude phase shift
2. **Swap o1/o2 connections** for instances with 180° rotation in the netlist

When a `straight_heater_meander` is placed with `transform = [-1,0,0,-1]` (180° rotation),
the physical ports swap sides. The nyancir records `o2→W2` (splitter side) and `o1→W4` (combiner side),
but the model expects light to enter o1. By swapping the connections at the netlist level,
the VCVS constraint `o2 = T × o1` works correctly.

### Why Y-matrix doesn't work here

The Y-matrix is mathematically correct for reciprocal devices, but the entries
$Y \propto 1/(1-T^2)$ oscillate when the phase $\phi$ changes (as during a current sweep).
This causes:
- Wild amplitude oscillations at internal nodes
- Very small detected signal at PDs
- PD1 and PD2 on top of each other (no complementary behavior)

The Y-matrix works fine for SAX models (mmi1x2, mmi2x2) because their parameters
don't change during a DC current sweep.

In [ ]:
# Identify flipped instances from the nyancir
print('Instance transforms:')
for key, data in nyancir.items():
    if isinstance(data, dict) and 'transform' in data and 'nets' in data:
        name = data.get('name', key.split(':')[-1])
        transform = data['transform']
        is_180 = transform[0] < 0 and transform[3] < 0
        model = data.get('model', data.get('type', ''))
        print(f'  {name:6s}: transform=[{transform[0]:2},{transform[3]:2}] '
              f'flipped_180={is_180}  model={model}')
        if is_180 and 'heater' in str(model).lower():
            nets = data['nets']
            print(f'         → NEEDS PORT SWAP: o1={nets.get("o1")}, o2={nets.get("o2")}')
            print(f'         → After swap:      o1={nets.get("o2")}, o2={nets.get("o1")}')